# 1.configurable_fields使用技巧

知识点讲解：configurable_fields 运行时配置链内部组件参数

1. configurable_fields 要解决的问题
   - 链一旦通过 | 组合完成，内部组件的参数就被固定了
   - 实际业务中常需要按请求动态调整（如不同用户使用不同 temperature）
   - 重复构建多套链代价高，configurable_fields 提供了运行时改参的能力
   - 与 bind() 的对比：bind() 是在构建时指定参数，configurable_fields() 是运行时才指定

2. configurable_fields 的工作机制与底层原理
   - 在组件上声明"哪些字段允许运行时配置"，并为每个字段分配唯一 id
   - 返回 RunnableConfigurableFields 对象，替代原组件参与链的组合
   - 调用时通过 config={"configurable": {id: value}} 传入新值
   - 框架在执行前根据 config 动态创建一个参数被替换的组件副本
   - 底层实现：调用 invoke() 时会先调用 _prepare() 预处理函数，该函数依据
     原有参数 + 配置的参数重新创建对应的组件进行覆盖

3. ConfigurableField 的三个参数
   - id: str - 配置键名（必填），是运行时 configurable 字典的键
   - name: str - 人类可读的字段名称（可选），用于 UI 展示或文档
   - description: str - 字段说明（可选），解释该参数的作用与取值影响

4. configurable_fields 与 bind() 的适用范围对比
   - configurable_fields() 和 bind() 非常接近，但可配置范围更广
   - 只要 Runnable 组件下有的所有属性，都可以通过 configurable_fields() 进行配置
   - bind() 只能配置**调用参数**（一般调用参数都和组件参数有关系）
   - 可通过 Runnable.__fields__.keys() 查看 configurable_fields() 支持配置哪些字段
     （父类属性也可以配置，但在这里不显示）

5. 两种传递配置的方式对比
   - with_config(configurable={...})：返回配置好的新链，适合复用同一配置多次调用
   - invoke(input, config={"configurable": {...}})：单次调用生效，适合每次都不同
   - 两者效果等价，前者更适合"配置固定、多次调用"的场景

6. temperature 参数的实际含义
   - 取值范围通常为 0 到 2，控制模型输出的随机性
   - temperature=0：输出最确定，相同输入几乎总是相同输出（适合抽取、分类）
   - temperature 较高：输出更多样、更有创造性（适合创作、头脑风暴）
   - 本示例通过生成随机整数来直观体现温度对输出稳定性的影响

7. configurable_fields 与 bind 的选择原则
   - bind()：构建链时固定参数，运行时不可变，写法简单
   - configurable_fields()：声明可配置点，运行时按需传值，灵活但需先声明
   - 只有预先声明为 configurable 的字段才能在运行时被修改

8. 数据流转与配置注入过程
   - chain.with_config(configurable={"llm_temperature": 0})
   - → 生成携带配置的新链（原链不受影响）
   - → invoke({"x": 1000}) 时，框架读取 configurable
   - → 用 temperature=0 替换 llm 的原始 temperature，生成组件副本
   - → {"x": 1000} → prompt → llm(temperature=0) → parser → str


In [ ]:
import dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import ConfigurableField
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
prompt = PromptTemplate.from_template("请生成一个小于{x}的随机整数")
llm = ChatOpenAI(model="deepseek-v4-pro").configurable_fields(
    temperature=ConfigurableField(
        id="llm_temperature",
        name="大语言模型的温度",
        description="温度越低，大语言模型生成的内容越确定，温度越高，生成内容越随机"
    )
)
chain = prompt | llm | StrOutputParser()
content = chain.invoke({"x": 1000})
print(content)
print("===========================")
with_config_chain = chain.with_config(configurable={"llm_temperature": 0})
content = with_config_chain.invoke({"x": 1000})
print(content)